In [1]:
import pandas as pd

# 1. Load data
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# 2. Create the buckets
df['staleness_bucket'] = pd.qcut(df['days_since_last_update'], q=4, duplicates='drop')

# 3. Group by bucket and aggregate
staleness_table = df.groupby('staleness_bucket', observed=False).agg(
    n=('content_id', 'count'),
    avg_clicks=('clicks_90d', 'mean'),
    avg_ctr=('ctr', 'mean')
)

print('--- Staleness Bucket Table ---')
print(staleness_table)
print('\nVerdict: MIXED')
print('Reasoning: Average clicks peak in the (20, 104] days bucket before dropping for highly stale content (>104 days).')

--- Staleness Bucket Table ---
                      n  avg_clicks   avg_ctr
staleness_bucket                             
(0.999, 20.0]     15866   13.568259  0.733422
(20.0, 104.0]     13816   19.114722  0.212029
(104.0, 373.0]      318   11.185535  2.377799

Verdict: MIXED
Reasoning: Average clicks peak in the (20, 104] days bucket before dropping for highly stale content (>104 days).


In [2]:
import pandas as pd

# Load dataset if running cell independently
if 'df' not in locals():
    df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# 1. Define logical SERP position boundaries (e.g., Top 3, Page 1 bottom, Page 2, Page 3+)
bins = [0, 3, 10, 20, 100]
labels = ['Top 3', 'Pos 4-10', 'Page 2', 'Page 3+']

df['position_bucket'] = pd.cut(df['avg_position'], bins=bins, labels=labels)

# 2. Group and aggregate
ctr_pos_table = df.groupby('position_bucket', observed=False).agg(
    n=('content_id', 'count'), 
    avg_ctr=('ctr', 'mean')
)

print("\n--- CTR vs Position Bucket Table ---")
print(ctr_pos_table)
print("\nVerdict: CONFIRMED")
print("Reasoning: CTR drops sharply as rank position increases (from ~2.71% for Top 3 down to ~0.21% for Page 3+).")


--- CTR vs Position Bucket Table ---
                     n   avg_ctr
position_bucket                 
Top 3             1141  2.714303
Pos 4-10         11842  0.651045
Page 2            7273  0.323443
Page 3+           8524  0.211705

Verdict: CONFIRMED
Reasoning: CTR drops sharply as rank position increases (from ~2.71% for Top 3 down to ~0.21% for Page 3+).


In [3]:
import pandas as pd

# Load dataset if running cell independently
if 'df' not in locals():
    df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# 1. Define what a "bad" CTR is for Page 1. 
PAGE_1_THRESHOLD = 10
BAD_CTR_THRESHOLD = 0.01 

# 2. Create a mask (a true/false filter) for the rows that meet our rule
rule_mask = (df['avg_position'] <= PAGE_1_THRESHOLD) & (df['ctr'] < BAD_CTR_THRESHOLD)

# 3. Create the three required columns. 
df['action_label'] = 'None'
df['reason_code'] = 'None'
df['score'] = 0.0

# 4. Apply the rule labels using .loc
df.loc[rule_mask, 'action_label'] = 'CTR-fix'
df.loc[rule_mask, 'reason_code'] = 'underperforming_ctr'

# 5. Calculate the score. 
df.loc[rule_mask, 'score'] = df.loc[rule_mask, 'impressions_90d'] 

# 6. Sort the dataframe so your worst offenders (highest score) are at the top!
baseline_queue = df[df['action_label'] != 'None'].sort_values(by='score', ascending=False)

# Let's look at your top 5
print(baseline_queue[['content_id', 'avg_position', 'ctr', 'impressions_90d', 'action_label', 'reason_code', 'score']].head())

                 content_id  avg_position  ctr  impressions_90d action_label  \
7445   content_c8e9d6ab9013           9.7  0.0           208678      CTR-fix   
23220  content_f986bd514b6e           6.6  0.0            22456      CTR-fix   
25462  content_825a9788af8d           5.6  0.0            16786      CTR-fix   
9443   content_8ba781dafa55           9.0  0.0            16156      CTR-fix   
12869  content_5d5653c4eb4f           5.7  0.0            15101      CTR-fix   

               reason_code     score  
7445   underperforming_ctr  208678.0  
23220  underperforming_ctr   22456.0  
25462  underperforming_ctr   16786.0  
9443   underperforming_ctr   16156.0  
12869  underperforming_ctr   15101.0  


In [4]:
import pandas as pd
import os

# Prepare baseline queue if running cell independently
if 'baseline_queue' not in locals():
    if 'df' not in locals():
        df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
    rule_mask = (df['avg_position'] <= 10) & (df['ctr'] < 0.01)
    df['action_label'] = 'None'
    df['reason_code'] = 'None'
    df['score'] = 0.0
    df.loc[rule_mask, 'action_label'] = 'CTR-fix'
    df.loc[rule_mask, 'reason_code'] = 'underperforming_ctr'
    df.loc[rule_mask, 'score'] = df.loc[rule_mask, 'impressions_90d']
    baseline_queue = df[df['action_label'] != 'None'].sort_values(by='score', ascending=False)

# Write to the specific outputs folder. index=False keeps it clean.
os.makedirs('../../outputs', exist_ok=True)
baseline_queue.to_csv('../../outputs/baseline_action_score.csv', index=False)
print("CSV successfully written to outputs!")

CSV successfully written to outputs!


In [1]:
import pandas as pd

# Prepare baseline queue if running cell independently
if 'baseline_queue' not in locals():
    if 'df' not in locals():
        df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
    rule_mask = (df['avg_position'] <= 10) & (df['ctr'] < 0.01)
    df['action_label'] = 'None'
    df['reason_code'] = 'None'
    df['score'] = 0.0
    df.loc[rule_mask, 'action_label'] = 'CTR-fix'
    df.loc[rule_mask, 'reason_code'] = 'underperforming_ctr'
    df.loc[rule_mask, 'score'] = df.loc[rule_mask, 'impressions_90d']
    baseline_queue = df[df['action_label'] != 'None'].sort_values(by='score', ascending=False)

# Print the columns you need to evaluate your top 10
columns_to_view = ['content_id', 'action_label', 'avg_position', 'ctr', 'impressions_90d']
print(baseline_queue[columns_to_view].head(10))

                 content_id action_label  avg_position  ctr  impressions_90d
7445   content_c8e9d6ab9013      CTR-fix           9.7  0.0           208678
23220  content_f986bd514b6e      CTR-fix           6.6  0.0            22456
25462  content_825a9788af8d      CTR-fix           5.6  0.0            16786
9443   content_8ba781dafa55      CTR-fix           9.0  0.0            16156
12869  content_5d5653c4eb4f      CTR-fix           5.7  0.0            15101
28588  content_847a841969a2      CTR-fix           7.4  0.0            14519
17362  content_c82bc0c24241      CTR-fix           4.3  0.0            13676
94     content_9983d31c53cb      CTR-fix           5.5  0.0             7737
2491   content_d3aaf7d5f2fc      CTR-fix           8.3  0.0             7732
17230  content_5195668f06db      CTR-fix           5.2  0.0             6635



content_c8e9d6ab9013: Action is CTR-fix. It's here due to Pos 9.7, 208,678 impressions, 0.0 CTR. It would be wrong if it's a zero-click query where Google directly answers the question on the search page.

content_f986bd514b6e: Action is CTR-fix. It's here due to Pos 6.6, 22,456 impressions, 0.0 CTR. It would be wrong if there's an intent mismatch (e.g., ranking for a brand name when users want a definition).

content_825a9788af8d: Action is CTR-fix. It's here due to Pos 5.6, 16,786 impressions, 0.0 CTR. It would be wrong if a massive video carousel is pushing this link visually below the fold.

content_8ba781dafa55: Action is CTR-fix. It's here due to Pos 9.0, 16,156 impressions, 0.0 CTR. It would be wrong if it's a utility page (like /login) that users navigate to via sitelinks on the homepage instead.

content_5d5653c4eb4f: Action is CTR-fix. It's here due to Pos 5.7, 15,101 impressions, 0.0 CTR. It would be wrong if the title includes an outdated year, meaning it just needs a date tweak, not a full rewrite.

content_847a841969a2: Action is CTR-fix. It's here due to Pos 7.4, 14,519 impressions, 0.0 CTR. It would be wrong if Google is serving this page for a highly specific branded search belonging to a competitor.

content_c82bc0c24241: Action is CTR-fix. It's here due to Pos 4.3, 13,676 impressions, 0.0 CTR. It would be wrong if the impressions spiked yesterday due to a viral event, but the click data pipeline hasn't caught up yet.

content_9983d31c53cb: Action is CTR-fix. It's here due to Pos 5.5, 7,737 impressions, 0.0 CTR. It would be wrong if it's a zero-click query where the meta description gives the full answer without needing a click.

content_d3aaf7d5f2fc: Action is CTR-fix. It's here due to Pos 8.3, 7,732 impressions, 0.0 CTR. It would be wrong if it's ranking for a transactional keyword when it's actually an informational blog post.

content_5195668f06db: Action is CTR-fix. It's here due to Pos 5.2, 6,635 impressions, 0.0 CTR. It would be wrong if a local map pack is stealing all the clicks for this specific query.